In [2]:

import fastcore as fc
import os
import math
import time
import inspect
import tiktoken
import numpy as np

from torch.distributed import init_process_group, destroy_process_group
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed as dist

from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F
from functools import partial
from HellaSwag import render_example, iterate_examples


/home/seif/Desktop/PycharmProjects/AI&ML dbrouke course/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


# AdamW

In [3]:
class AdamW:
    def __init__(self, params, lr, wd=0., beta1=0.9, beta2=0.99, eps=1e-5):
        params = list(params)
        fc.store_attr() # Just to store all the attributes
        self.i = 0

    def step(self):
        with torch.no_grad():
            for p in self.params:
                self.opt_step(p)
        self.i += 1

    def opt_step(self,p):
        if not hasattr(p,'grad_avg'): p.grad_avg = torch.zeros_like(p.grad.data)
        if not hasattr(p, 'grad_sqr_avg'): p.grad_sqr_avg = torch.zeros_like(p.grad.data)
        self.grad_avg.lerp_(p.grad, 1 - self.beta1)
        self.grad_sqr_avg.lerp_(p.grad.square(), self.beta2)
        unbiased_grad_avg = self.grad_avg / (1 - self.beta1 ** (self.i+1) )
        unbiased_grad_sqr_avg = self.grad_sqr_avg / (1 - self.beta2 ** (self.i+1) )
        update = unbiased_grad_avg / (unbiased_grad_sqr_avg.sqrt() + self.eps)
        if self.wd : update += self.wd * p.data
        p.data -= self.lr * update

    def zero_grad(self):
        for p in self.params : p.grad.data.zero_()

# Muon Optimizer

In [4]:
class Muon_optim:
    def __init__(self, params, lr, grad_avg, v_mean_avg, beta2, red_dim, wd=0., steps=5 , beta1=0.9, Nesterov=True):
        params = list(params)
        fc.store_attr()
        self.i = 0

    def step(self):
        with torch.no_grad():
            for p in self.params: self.opt_step(p)
        self.i += 1

    def zero_grad(self):
        for p in self.params : p.grad.data.zero_()

    def opt_step(self, p):
        g = p.grad.data

        # Nesterov momentum
        if not hasattr(p,'grad_avg'): p.grad_avg = torch.zeros_like(g)
        self.grad_avg.lerp_(g, 1 - self.beta1)
        unbiased_grad_avg = self.grad_avg  / (1 - self.beta1 ** (self.i+1) )
        g = g.lerp_(unbiased_grad_avg, self.beta1) if self.Nesterov else unbiased_grad_avg

        # Normalization using Norm
        target = g.norm(dim=(-1,-2), keepdim=True) * (g.size(-2)**-0.5)
        row_norm = g.norm(dim=(-1), keepdim=True)
        g = g * (target / row_norm)

        # Newton sched
        a, b, c = 3.4445, -4.7750, 2.0315
        for _ in range(self.steps):
            A = g @ g.mT                # mT transposes the last 2 dims
            B = b * A + c * (A @ A)
            g = a * g + B @ g

        # Muon+ normalization
        targ_norm = min(g.size(-2), g.size(-1))  ** 0.5
        current_norm = g.norm(dim=(-1,-2), keepdim=True)
        g = g * (targ_norm / current_norm)

        # Variance Reduction
        v_mean = g.square().mean(dim=self.red_dim, keepdim=True)
        red_dim_sz = g.size(self.red_dim)
        v_norm_sq = v_mean.sum(dim=(-1,-2), keepdim=True) * red_dim_sz
        v_norm = v_norm_sq.sqrt()
        if not hasattr(p, 'v_mean_avg'): p.v_mean_avg = torch.zeros_like(v_norm)
        self.v_mean_avg.lerp_(v_norm, 1 - self.beta2)
        unbiased_v_mean_avg = self.v_mean_avg  / (1 - self.beta2 ** (self.i+1))
        step_sz = unbiased_v_mean_avg.rsqrt()
        scaled_sq_sum = (v_mean * red_dim_sz) * step_sz.square()
        v_norm_new = scaled_sq_sum.sum(dim=(-1,-2), keepdim=True).sqrt()
        final_scale = step_sz * (v_norm / v_norm_new)
        g = g * final_scale

        # Update
        mask = (g * unbiased_grad_avg) >= 0   # TODO: DO I USE THE UNBIASED OR THE NORMAL ONE
        update = g + self.wd * p.data * mask if self.wd else g
        p.data.sub_(self.lr * update) # Make it in place

    ADD CLAMPS FOR STABILITY:
        row_norm = g.norm(dim=(-1), keepdim=True).clamp(min=1e-6) # Clamp for stability


        if g.size(-1) > g.size(-2):     # if it is wider Matrix, just for optimization
            for _ in range(steps):
                A = g @ g.mT                # mT transposes the last 2 dims
                B = b * A + c * (A @ A)
                g = a * g + B @ g
        else:
            for _ in range(steps):
                A = g.mT @ g   #
                B = b * A + c * (A @ A)
                g = a * g + g @ B


    WHY 5 ITERATIONS:
    ─────────────────────────────────────────
    each iteration makes X closer to orthogonal
    5 iterations is enough for good approximation
    more iterations = diminishing returns ✓

# MuonAdamW

## Create it

In [ ]:
class MuonAdamW:
    def __init__(self, params, lr,  beta2, red_dim, wd=0., steps=5 , beta1=0.9, Nesterov=True, eps=1e-7):
        params = list(params)
        fc.store_attr()
        self.i = 0
    def step(self):
        with torch.no_grad():
            for p in self.params: self.opt_step(p)
        self.i += 1

    def zero_grad(self):
        for p in self.params :
            if p.grad is not None : p.grad.data.zero_()

    def opt_step(self, p):
        if p.dim() < 2 :
            if not hasattr(p,'grad_avg'): p.grad_avg = torch.zeros_like(p.grad.data)
            if not hasattr(p, 'grad_sqr_avg'): p.grad_sqr_avg = torch.zeros_like(p.grad.data)
            p.grad_avg.lerp_(p.grad, 1 - self.beta1)
            p.grad_sqr_avg.lerp_(p.grad.square(), 1 - self.beta2)
            unbiased_grad_avg = p.grad_avg / (1 - self.beta1 ** (self.i+1) )
            unbiased_grad_sqr_avg = p.grad_sqr_avg / (1 - self.beta2 ** (self.i+1) )
            update = unbiased_grad_avg / (unbiased_grad_sqr_avg.sqrt() + self.eps)
            if self.wd : update += self.wd * p.data
            p.data.sub_(self.lr * update)
        else:
            g = p.grad.data

            # Nesterov momentum
            if not hasattr(p,'grad_avg'): p.grad_avg = torch.zeros_like(g)
            p.grad_avg.lerp_(g, 1 - self.beta1)
            unbiased_grad_avg = p.grad_avg  / (1 - self.beta1 ** (self.i+1) )
            g = g.lerp(unbiased_grad_avg, self.beta1) if self.Nesterov else unbiased_grad_avg

            # Normalization using Norm
            target = g.norm(dim=(-1,-2), keepdim=True) * (g.size(-2)**-0.5)
            row_norm = g.norm(dim=(-1), keepdim=True)
            g = g * (target / row_norm)

            # Frobenius norm
            g /= (g.norm(dim=(-1,-2), keepdim=True) * 1.01 + self.eps)  # 1.01 is just safety so that everything is <1 and <-1 and not =

            # Newton sched
            a, b, c = 3.4445, -4.7750, 2.0315
            for _ in range(self.steps):
                A = g @ g.mT                # mT transposes the last 2 dims
                B = b * A + c * (A @ A)
                g = a * g + B @ g

            # Muon+ normalization
            targ_norm = min(g.size(-2), g.size(-1))  ** 0.5
            current_norm = g.norm(dim=(-1,-2), keepdim=True)
            g = g * (targ_norm / current_norm)

            # Variance Reduction
            v_mean = g.square().mean(dim=self.red_dim, keepdim=True)
            red_dim_sz = g.size(self.red_dim)
            v_norm_sq = v_mean.sum(dim=(-1,-2), keepdim=True) * red_dim_sz
            v_norm = v_norm_sq.sqrt()
            if not hasattr(p, 'v_mean_avg'): p.v_mean_avg = torch.zeros_like(v_mean)
            p.v_mean_avg.lerp_(v_mean, 1 - self.beta2)
            unbiased_v_mean_avg = p.v_mean_avg  / (1 - self.beta2 ** (self.i+1))
            step_sz = (unbiased_v_mean_avg+self.eps).rsqrt()
            scaled_sq_sum = (v_mean * red_dim_sz) * step_sz.square()
            v_norm_new = scaled_sq_sum.sum(dim=(-1,-2), keepdim=True).sqrt()
            final_scale = step_sz * (v_norm / v_norm_new)
            g = g * final_scale

            # Update
            mask = (g * unbiased_grad_avg) >= 0   # TODO: DO I USE THE UNBIASED OR THE NORMAL ONE
            update = g + self.wd * p.data * mask if self.wd else g
            p.data.sub_(self.lr * update) # Make it in place




# Visualize the steps of MuonAdamW on a sample gradient matrix

In [ ]:
import torch
torch.manual_seed(0)
torch.set_printoptions(precision=3, sci_mode=False)

def show(name, t):
    print(f"\n--- {name} ---")
    print(t)

# 4 output neurons, 6 input features — deliberately uneven row norms
g = torch.tensor([
    [8.0,  0.0,  0.0,  0.0,  0.0,  0.0],   # dominant row
    [0.01, 0.01, 0.0,  0.0,  0.0,  0.0],   # nearly dead row
    [1.0,  1.0,  1.0,  0.0,  0.0,  0.0],
    [0.3, -0.3,  0.3, -0.3,  0.3, -0.3],
])
show("RAW GRADIENT", g)
print("row norms:", g.norm(dim=-1))
print("total frobenius norm :", g.norm().item())


# STEP 1: row equilibration
target = g.norm(dim=(-2,-1), keepdim=True) * (g.size(-2) ** -0.5)
row_norm = g.norm(dim=-1, keepdim=True)
g = g * (target / row_norm)
show("AFTER ROW EQUILIBRATION", g)
print("row norms (now equal):", g.norm(dim=-1))
print("total frobenius norm (unchanged):", g.norm().item())

# STEP 2: frobenius normalization (with 1.01 safety margin)
g = g / (g.norm(dim=(-2,-1), keepdim=True) * 1.01 + 1e-6)
show("AFTER FROBENIUS NORM", g)
print("total frobenius norm:", g.norm().item())

# STEP 3: Newton-Schulz
a, b, c = 3.4445, -4.7750, 2.0315
for step in range(5):
    A = g @ g.mT
    B = b * A + c * (A @ A)
    g = a * g + B @ g
    show(f"NS iter {step+1} singular values", g)

# STEP 4: Muon+ renormalization
targ_norm = min(g.size(-2), g.size(-1)) ** 0.5
current_norm = g.norm(dim=(-2,-1), keepdim=True)
print(f"\ntarget norm = sqrt(min(out,in)) = {targ_norm:.3f}, current = {current_norm.item():.3f}")
show('ROWS NORMS', g.norm(dim=(-1, -2), keepdim=True))
g = g * (targ_norm / current_norm)
show("AFTER MUON+ RENORM", g)
print("frobenius norm now:", g.norm().item())

# STEP 5: variance reduction (per-row)
red_dim = -1
v_mean = g.square().mean(dim=red_dim, keepdim=True)
show("PER-ROW MEAN SQUARE (v_mean)", v_mean)

step_sz = (v_mean + 1e-8).rsqrt()
show("PER-ROW STEP SIZE (1/sqrt(v_mean))", step_sz)

red_dim_sz = g.size(red_dim)
v_norm = (v_mean.sum() * red_dim_sz).sqrt()
scaled_sq_sum = (v_mean * red_dim_sz) * step_sz.square()
v_norm_new = scaled_sq_sum.sum().sqrt()
final_scale = step_sz * (v_norm / v_norm_new)
show(" PER-ROW (v_norm / v_norm_new)", final_scale)

show("FINAL PER-ROW SCALE", final_scale)

norm_before = g.norm().item()
show("BEFORE VARIANCE REDUCTION", g)
g = g * final_scale
show("AFTER VARIANCE REDUCTION", g)
print(f"frobenius norm before: {norm_before:.4f}  after: {g.norm().item():.4f}  ")

# STEP 6: cautious weight decay mask
fake_grad_momentum = torch.randn_like(g) * 0.1
mask = (g * fake_grad_momentum) >= 0
show("MASK (True = update and momentum agree in sign)", mask)

    ─────────────────────────────────────────
    row equilibration:   fixes the spread ACROSS rows (not within one),
                          so 5 fixed NS steps are actually enough
    frobenius norm:       shrinks the WHOLE matrix into NS's working range,
                          1.01 is just floating-point insurance
    Newton-Schulz:        the real orthogonalizer — pushes every singular
                          value toward 1, equivalently A=g@g.T toward identity
    Muon+:                a magnitude-only cleanup AFTER NS, snapping the
                          one aggregate number (total frobenius norm) to what
                          perfect convergence would have given, patching NS's
                          5-step imperfection without touching direction
    variance reduction:   Adam's trick, just per-row instead of per-element

    each step fixes a DIFFERENT failure mode of the previous one — none of
    them do the same job twice.

## Create a more general MuonAdamW that can handle multiple parameter groups and weight decay

In [ ]:
class MuonAdamW:
    def __init__(self, params, lr, beta2, red_dim, steps=5 , beta1=0.9, Nesterov=True, eps=1e-7):
        fc.store_attr()
        self.i = 0

    def step(self, lr=None):
        if lr is None: lr = self.lr
        with torch.no_grad():
            for g in self.params:
                for p in g:   self.opt_step(p['params'], p['weight_decay'], lr)
        self.i += 1

    def zero_grad(self):
        for g in self.params :
            for p in g :
                if p.grad is not None : p.grad.data.zero_()

    def opt_step(self, p, wd=None, lr):
        if p.dim() < 2 :
            if not hasattr(p,'grad_avg'): p.grad_avg = torch.zeros_like(p.grad.data)
            if not hasattr(p, 'grad_sqr_avg'): p.grad_sqr_avg = torch.zeros_like(p.grad.data)
            p.grad_avg.lerp_(p.grad, 1 - self.beta1)
            p.grad_sqr_avg.lerp_(p.grad.square(), 1 - self.beta2)
            unbiased_grad_avg = p.grad_avg / (1 - self.beta1 ** (self.i+1) )
            unbiased_grad_sqr_avg = p.grad_sqr_avg / (1 - self.beta2 ** (self.i+1) )
            update = unbiased_grad_avg / (unbiased_grad_sqr_avg.sqrt() + self.eps)
            if wd : update += wd * p.data
            p.data.sub_(lr * update)
        else:
            g = p.grad.data

            # Nesterov momentum
            if not hasattr(p,'grad_avg'): p.grad_avg = torch.zeros_like(g)
            p.grad_avg.lerp_(g, 1 - self.beta1)
            unbiased_grad_avg = p.grad_avg  / (1 - self.beta1 ** (self.i+1) )
            g = g.lerp(unbiased_grad_avg, self.beta1) if self.Nesterov else unbiased_grad_avg

            # Normalization using Norm
            target = g.norm(dim=(-1,-2), keepdim=True) * (g.size(-2)**-0.5)
            row_norm = g.norm(dim=(-1), keepdim=True)
            g = g * (target / row_norm)

            # Frobenius norm
            g /= (g.norm(dim=(-1,-2), keepdim=True) * 1.01 + self.eps)  # 1.01 is just safety so that everything is <1 and <-1 and not =

            # Newton sched
            a, b, c = 3.4445, -4.7750, 2.0315
            for _ in range(self.steps):
                A = g @ g.mT                # mT transposes the last 2 dims
                B = b * A + c * (A @ A)
                g = a * g + B @ g

            # Muon+ normalization
            targ_norm = min(g.size(-2), g.size(-1))  ** 0.5
            current_norm = g.norm(dim=(-1,-2), keepdim=True)
            g = g * (targ_norm / current_norm)

            # Variance Reduction
            v_mean = g.square().mean(dim=self.red_dim, keepdim=True)
            red_dim_sz = g.size(self.red_dim)
            v_norm_sq = v_mean.sum(dim=(-1,-2), keepdim=True) * red_dim_sz
            v_norm = v_norm_sq.sqrt()
            if not hasattr(p, 'v_mean_avg'): p.v_mean_avg = torch.zeros_like(v_mean)
            p.v_mean_avg.lerp_(v_mean, 1 - self.beta2)
            unbiased_v_mean_avg = p.v_mean_avg  / (1 - self.beta2 ** (self.i+1))
            step_sz = (unbiased_v_mean_avg+self.eps).rsqrt()
            scaled_sq_sum = (v_mean * red_dim_sz) * step_sz.square()
            v_norm_new = scaled_sq_sum.sum(dim=(-1,-2), keepdim=True).sqrt()
            final_scale = step_sz * (v_norm / v_norm_new)
            g = g * final_scale

            # Update
            mask = (g * unbiased_grad_avg) >= 0   # TODO: DO I USE THE UNBIASED OR THE NORMAL ONE
            update = g + wd * p.data * mask if wd else g
            p.data.sub_(lr * update) # Make it in place

# Creating RoPE

In [ ]:
def create_angles(head_dim, base=10000, sequence_length=100):
    dim = head_dim // 2
    x = torch.arange(dim)
    y = base  ** (- x / dim )
    positions = torch.arange(sequence_length)
    angles = positions[:, None] * y[None, :]
    return angles

def rotation(x, angles):
    x1 = x[...,:x.size(-1)//2]
    x2 = x[...,x.size(-1)//2:]

    sin_angles = angles.sin()
    cos_angles = angles.cos()

    # Applying the rotation matrix
    y1 = x1 * cos_angles - x2 * sin_angles
    y2 = x1 * sin_angles + x2 * cos_angles

    return torch.cat([y1, y2], dim=-1)

In [88]:
###________________________ CREATING THE RoPE POSITIONAL ENCODING ______________________

class RoPE(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        head_dim = config.n_embd // config.n_head
        inv_freq = self._create_inv_freq(head_dim, config.base)
        self.register_buffer('inv_freq', inv_freq)

    def _create_inv_freq(self, head_dim, base=10000):
        dim = head_dim // 2
        x = torch.arange(dim, dtype=torch.float32)
        inv_freq = base ** (- x / dim )
        return inv_freq

    def _create_angles(self, inv_freq, sequence_length=100):
        positions = torch.arange(sequence_length,device=inv_freq.device,dtype=torch.float32)
        angles = positions[:, None] * inv_freq[None, :]
        return angles

    def _rotation(self, x, angles):
        x1 = x[...,:x.shape[-1]//2]
        x2 = x[...,x.shape[-1]//2:]

        sin_angles = angles.sin()
        cos_angles = angles.cos()

        # Applying the rotation matrix
        y1 = x1 * cos_angles - x2 * sin_angles
        y2 = x1 * sin_angles + x2 * cos_angles

        return torch.cat([y1, y2], dim=-1)

    def forward(self, x):
        seq_len = x.shape[-2]
        angles = self._create_angles(self.inv_freq, seq_len)
        angles = angles.unsqueeze(0)
        return self._rotation(x, angles)

In [89]:
x = torch.randn(128, 32, 8) # (bs*n_heads, sl, Emb)
class config: pass
configs = config()
configs.n_embd = 64
configs.base = 10000
configs.n_head = 8
configs.block_size = 32
rope = RoPE(configs)

In [90]:
rope(x).shape

torch.Size([128, 32, 8])

In [91]:
rope.inv_freq.shape

torch.Size([4])

In [92]:
rope.inv_freq

tensor([1.0000, 0.1000, 0.0100, 0.0010])

    fddseif@seif-ASUS-TUF-Gaming-A15-FA507NVR-FA507NVR:~$ sudo apt update && sudo apt install gnome-network-displays
    [sudo] password for seif:
    Get:1 https://packages.microsoft.com/repos/code stable InRelease [3,590 B]
    Get:2 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
    Get:3 https://packages.microsoft.com/repos/code stable/main amd64 Packages [26.4 kB]
    Hit:4 https://ppa.launchpadcontent.net/ubuntuhandbook1/conkymanager2/ubuntu noble InRelease
    Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,581 B]
    Get:6 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [856 kB]
    Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [1,684 kB]
    Hit:8 https://repo.huaweicloud.com/ubuntu noble InRelease
    Get:9 https://repo.huaweicloud.com/ubuntu noble-updates InRelease [126 kB]
    Hit:10 https://us-central1-apt.pkg.dev/projects/antigravity-auto-updater-dev antigravity-debian InRelease
    Get:12 https://repo.huaweicloud.com/ubuntu noble-backports InRelease [126 kB]
    Get:13 http://security.ubuntu.com/ubuntu noble-security/main Translation-en [192 kB]
    Get:14 http://security.ubuntu.com/ubuntu noble-security/main amd64 Components [44.8 kB]
    Get:15 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [1,166 kB]
    Get:16 https://repo.huaweicloud.com/ubuntu noble-updates/main amd64 Packages [1,114 kB]
    Hit:11 https://packagecloud.io/github/git-lfs/ubuntu noble InRelease
    Get:17 http://security.ubuntu.com/ubuntu noble-security/restricted Translation-en [268 kB]
    Get:18 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,180 kB]
    Get:19 http://security.ubuntu.com/ubuntu noble-security/universe Translation-en [233 kB]
    Get:20 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Components [76.3 kB]
    Get:21 https://repo.huaweicloud.com/ubuntu noble-updates/main Translation-en [272 kB]
    Get:22 https://repo.huaweicloud.com/ubuntu noble-updates/main amd64 Components [180 kB]
    Get:23 https://repo.huaweicloud.com/ubuntu noble-updates/restricted amd64 Packages [1,256 kB]
    Get:24 https://repo.huaweicloud.com/ubuntu noble-updates/restricted Translation-en [286 kB]
    Get:25 https://repo.huaweicloud.com/ubuntu noble-updates/universe amd64 Packages [1,661 kB]
    Get:26 https://repo.huaweicloud.com/ubuntu noble-updates/universe Translation-en [328 kB]
    Get:27 https://repo.huaweicloud.com/ubuntu noble-updates/universe amd64 Components [388 kB]
    Get:28 https://repo.huaweicloud.com/ubuntu noble-updates/multiverse amd64 Components [940 B]
    Get:29 https://repo.huaweicloud.com/ubuntu noble-backports/main amd64 Components [5,788 B]
    Get:30 https://repo.huaweicloud.com/ubuntu noble-backports/universe amd64 Components [10.6 kB]
    Fetched 11.6 MB in 4s (2,792 kB/s)
    Reading package lists... Done
    Building dependency tree... Done
    Reading state information... Done
    20 packages can be upgraded. Run 'apt list --upgradable' to see them.
    Reading package lists... Done
    Building dependency tree... Done
    Reading state information... Done
    The following additional packages will be installed:
      gstreamer1.0-plugins-bad imagemagick-6-common libavahi-gobject0 libavtp0
      libdirectfb-1.7-7t64 libfluidsynth3 libgstrtspserver-1.0-0
      libinstpatch-1.0-2 libjxr-tools libjxr0t64 liblqr-1-0 liblrdf0 libltc11
      libmagickcore-6.q16-7-extra libmagickcore-6.q16-7t64
      libmagickwand-6.q16-7t64 libmfx1 libmjpegutils-2.1-0t64 libmodplug1
      libmpeg2encpp-2.1-0t64 libmplex2-2.1-0t64 libneon27t64 libopenh264-7
      libopenni2-0 libqrencode4 libraptor2-0 libsoundtouch1 libspandsp2t64
      libsrtp2-1 libvo-aacenc0 libvo-amrwbenc0 libwildmidi2 libyajl2 libzbar0t64
      libzxing3 timgm6mb-soundfont
    Suggested packages:
      frei0r-plugins libdirectfb-extra liblrdf0-dev inkscape raptor2-utils
      libwildmidi-config fluid-soundfont-gm
    The following NEW packages will be installed:
      gnome-network-displays gstreamer1.0-plugins-bad imagemagick-6-common
      libavahi-gobject0 libavtp0 libdirectfb-1.7-7t64 libfluidsynth3
      libgstrtspserver-1.0-0 libinstpatch-1.0-2 libjxr-tools libjxr0t64 liblqr-1-0
      liblrdf0 libltc11 libmagickcore-6.q16-7-extra libmagickcore-6.q16-7t64
      libmagickwand-6.q16-7t64 libmfx1 libmjpegutils-2.1-0t64 libmodplug1
      libmpeg2encpp-2.1-0t64 libmplex2-2.1-0t64 libneon27t64 libopenh264-7
      libopenni2-0 libqrencode4 libraptor2-0 libsoundtouch1 libspandsp2t64
      libsrtp2-1 libvo-aacenc0 libvo-amrwbenc0 libwildmidi2 libyajl2 libzbar0t64
      libzxing3 timgm6mb-soundfont
    0 upgraded, 37 newly installed, 0 to remove and 20 not upgraded.
    Need to get 18.7 MB of archives.
    After this operation, 65.9 MB of additional disk space will be used.
    Do you want to continue? [Y/n] y
    Get:1 https://repo.huaweicloud.com/ubuntu noble/universe amd64 liblqr-1-0 amd64 0.4.2-2.1build2 [28.5 kB]
    Get:2 https://repo.huaweicloud.com/ubuntu noble/universe amd64 imagemagick-6-common all 8:6.9.12.98+dfsg1-5.2build2 [69.5 kB]
    Get:3 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libmagickcore-6.q16-7t64 amd64 8:6.9.12.98+dfsg1-5.2build2 [1,811 kB]
    Get:4 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libmagickwand-6.q16-7t64 amd64 8:6.9.12.98+dfsg1-5.2build2 [318 kB]
    Get:5 https://repo.huaweicloud.com/ubuntu noble-updates/main amd64 libavahi-gobject0 amd64 0.8-13ubuntu6.2 [17.9 kB]
    Get:6 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libavtp0 amd64 0.2.0-1build1 [6,414 B]
    Get:7 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libdirectfb-1.7-7t64 amd64 1.7.7-11.1ubuntu2 [1,035 kB]
    Get:8 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libinstpatch-1.0-2 amd64 1.1.6-1build2 [251 kB]
    Get:9 https://repo.huaweicloud.com/ubuntu noble/universe amd64 timgm6mb-soundfont all 1.3-5 [5,427 kB]
    Get:10 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libfluidsynth3 amd64 2.3.4-1build3 [249 kB]
    Get:11 https://repo.huaweicloud.com/ubuntu noble/main amd64 libyajl2 amd64 2.1.0-5build1 [20.2 kB]
    Get:12 https://repo.huaweicloud.com/ubuntu noble-updates/main amd64 libraptor2-0 amd64 2.0.16-3ubuntu0.1 [165 kB]
    Get:13 https://repo.huaweicloud.com/ubuntu noble/universe amd64 liblrdf0 amd64 0.6.1-4build1 [18.5 kB]
    Get:14 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libltc11 amd64 1.3.2-1build1 [13.0 kB]
    Get:15 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libmfx1 amd64 22.5.4-1 [3,124 kB]
    Get:16 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libmjpegutils-2.1-0t64 amd64 1:2.1.0+debian-8.1build1 [25.5 kB]
    Get:17 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libmodplug1 amd64 1:0.8.9.0-3build1 [166 kB]
    Get:18 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libmpeg2encpp-2.1-0t64 amd64 1:2.1.0+debian-8.1build1 [75.6 kB]
    Get:19 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libmplex2-2.1-0t64 amd64 1:2.1.0+debian-8.1build1 [46.1 kB]
    Get:20 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libneon27t64 amd64 0.33.0-1.1build3 [102 kB]
    Get:21 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libopenh264-7 amd64 2.4.1+dfsg-1 [409 kB]
    Get:22 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libopenni2-0 amd64 2.2.0.33+dfsg-18 [370 kB]
    Get:23 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libqrencode4 amd64 4.1.1-1build2 [25.0 kB]
    Get:24 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libsoundtouch1 amd64 2.3.2+ds1-1build1 [60.5 kB]
    Get:25 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libspandsp2t64 amd64 0.0.6+dfsg-2.1build1 [311 kB]
    Get:26 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libsrtp2-1 amd64 2.5.0-3build1 [41.9 kB]
    Get:27 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libvo-aacenc0 amd64 0.1.3-2build1 [67.8 kB]
    Get:28 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libvo-amrwbenc0 amd64 0.1.3-2build1 [76.7 kB]
    Get:29 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libwildmidi2 amd64 0.4.3-1build3 [68.5 kB]
    Get:30 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libzbar0t64 amd64 0.23.93-4build3 [123 kB]
    Get:31 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libzxing3 amd64 2.2.1-3 [583 kB]
    Get:32 https://repo.huaweicloud.com/ubuntu noble/universe amd64 gstreamer1.0-plugins-bad amd64 1.24.2-1ubuntu4 [3,081 kB]
    Get:33 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libgstrtspserver-1.0-0 amd64 1.24.2-1 [155 kB]
    Get:34 https://repo.huaweicloud.com/ubuntu noble/universe amd64 gnome-network-displays amd64 0.92.1-2build2 [115 kB]
    Get:35 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libjxr0t64 amd64 1.2~git20170615.f752187-5.1ubuntu2 [181 kB]
    Get:36 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libjxr-tools amd64 1.2~git20170615.f752187-5.1ubuntu2 [15.8 kB]
    Get:37 https://repo.huaweicloud.com/ubuntu noble/universe amd64 libmagickcore-6.q16-7-extra amd64 8:6.9.12.98+dfsg1-5.2build2 [70.1 kB]
    Fetched 18.7 MB in 14s (1,374 kB/s)
    Extracting templates from packages: 100%
    Selecting previously unselected package liblqr-1-0:amd64.
    (Reading database ... 281399 files and directories currently installed.)
    Preparing to unpack .../00-liblqr-1-0_0.4.2-2.1build2_amd64.deb ...
    Unpacking liblqr-1-0:amd64 (0.4.2-2.1build2) ...
    Selecting previously unselected package imagemagick-6-common.
    Preparing to unpack .../01-imagemagick-6-common_8%3a6.9.12.98+dfsg1-5.2build2_al
    l.deb ...
    Unpacking imagemagick-6-common (8:6.9.12.98+dfsg1-5.2build2) ...
    Selecting previously unselected package libmagickcore-6.q16-7t64:amd64.
    Preparing to unpack .../02-libmagickcore-6.q16-7t64_8%3a6.9.12.98+dfsg1-5.2build
    2_amd64.deb ...
    Unpacking libmagickcore-6.q16-7t64:amd64 (8:6.9.12.98+dfsg1-5.2build2) ...
    Selecting previously unselected package libmagickwand-6.q16-7t64:amd64.
    Preparing to unpack .../03-libmagickwand-6.q16-7t64_8%3a6.9.12.98+dfsg1-5.2build
    2_amd64.deb ...
    Unpacking libmagickwand-6.q16-7t64:amd64 (8:6.9.12.98+dfsg1-5.2build2) ...
    Selecting previously unselected package libavahi-gobject0:amd64.
    Preparing to unpack .../04-libavahi-gobject0_0.8-13ubuntu6.2_amd64.deb ...
    Unpacking libavahi-gobject0:amd64 (0.8-13ubuntu6.2) ...
    Selecting previously unselected package libavtp0:amd64.
    Preparing to unpack .../05-libavtp0_0.2.0-1build1_amd64.deb ...
    Unpacking libavtp0:amd64 (0.2.0-1build1) ...
    Selecting previously unselected package libdirectfb-1.7-7t64:amd64.
    Preparing to unpack .../06-libdirectfb-1.7-7t64_1.7.7-11.1ubuntu2_amd64.deb ...
    Unpacking libdirectfb-1.7-7t64:amd64 (1.7.7-11.1ubuntu2) ...
    Selecting previously unselected package libinstpatch-1.0-2:amd64.
    Preparing to unpack .../07-libinstpatch-1.0-2_1.1.6-1build2_amd64.deb ...
    Unpacking libinstpatch-1.0-2:amd64 (1.1.6-1build2) ...
    Selecting previously unselected package timgm6mb-soundfont.
    Preparing to unpack .../08-timgm6mb-soundfont_1.3-5_all.deb ...
    Unpacking timgm6mb-soundfont (1.3-5) ...
    Selecting previously unselected package libfluidsynth3:amd64.
    Preparing to unpack .../09-libfluidsynth3_2.3.4-1build3_amd64.deb ...
    Unpacking libfluidsynth3:amd64 (2.3.4-1build3) ...
    Selecting previously unselected package libyajl2:amd64.
    Preparing to unpack .../10-libyajl2_2.1.0-5build1_amd64.deb ...
    Unpacking libyajl2:amd64 (2.1.0-5build1) ...
    Selecting previously unselected package libraptor2-0:amd64.
    Preparing to unpack .../11-libraptor2-0_2.0.16-3ubuntu0.1_amd64.deb ...
    Unpacking libraptor2-0:amd64 (2.0.16-3ubuntu0.1) ...
    Selecting previously unselected package liblrdf0:amd64.
    Preparing to unpack .../12-liblrdf0_0.6.1-4build1_amd64.deb ...
    Unpacking liblrdf0:amd64 (0.6.1-4build1) ...
    Selecting previously unselected package libltc11:amd64.
    Preparing to unpack .../13-libltc11_1.3.2-1build1_amd64.deb ...
    Unpacking libltc11:amd64 (1.3.2-1build1) ...
    Selecting previously unselected package libmfx1:amd64.
    Preparing to unpack .../14-libmfx1_22.5.4-1_amd64.deb ...
    Unpacking libmfx1:amd64 (22.5.4-1) ...
    Selecting previously unselected package libmjpegutils-2.1-0t64:amd64.
    Preparing to unpack .../15-libmjpegutils-2.1-0t64_1%3a2.1.0+debian-8.1build1_amd
    64.deb ...
    Unpacking libmjpegutils-2.1-0t64:amd64 (1:2.1.0+debian-8.1build1) ...
    Selecting previously unselected package libmodplug1:amd64.
    Preparing to unpack .../16-libmodplug1_1%3a0.8.9.0-3build1_amd64.deb ...
    Unpacking libmodplug1:amd64 (1:0.8.9.0-3build1) ...
    Selecting previously unselected package libmpeg2encpp-2.1-0t64:amd64.
    Preparing to unpack .../17-libmpeg2encpp-2.1-0t64_1%3a2.1.0+debian-8.1build1_amd
    64.deb ...
    Unpacking libmpeg2encpp-2.1-0t64:amd64 (1:2.1.0+debian-8.1build1) ...
    Selecting previously unselected package libmplex2-2.1-0t64:amd64.
    Preparing to unpack .../18-libmplex2-2.1-0t64_1%3a2.1.0+debian-8.1build1_amd64.d
    eb ...
    Unpacking libmplex2-2.1-0t64:amd64 (1:2.1.0+debian-8.1build1) ...
    Selecting previously unselected package libneon27t64:amd64.
    Preparing to unpack .../19-libneon27t64_0.33.0-1.1build3_amd64.deb ...
    Unpacking libneon27t64:amd64 (0.33.0-1.1build3) ...
    Selecting previously unselected package libopenh264-7:amd64.
    Preparing to unpack .../20-libopenh264-7_2.4.1+dfsg-1_amd64.deb ...
    Unpacking libopenh264-7:amd64 (2.4.1+dfsg-1) ...
    Selecting previously unselected package libopenni2-0:amd64.
    Preparing to unpack .../21-libopenni2-0_2.2.0.33+dfsg-18_amd64.deb ...
    Unpacking libopenni2-0:amd64 (2.2.0.33+dfsg-18) ...
    Selecting previously unselected package libqrencode4:amd64.
    Preparing to unpack .../22-libqrencode4_4.1.1-1build2_amd64.deb ...
    Unpacking libqrencode4:amd64 (4.1.1-1build2) ...
    Selecting previously unselected package libsoundtouch1:amd64.
    Preparing to unpack .../23-libsoundtouch1_2.3.2+ds1-1build1_amd64.deb ...
    Unpacking libsoundtouch1:amd64 (2.3.2+ds1-1build1) ...
    Selecting previously unselected package libspandsp2t64:amd64.
    Preparing to unpack .../24-libspandsp2t64_0.0.6+dfsg-2.1build1_amd64.deb ...
    Unpacking libspandsp2t64:amd64 (0.0.6+dfsg-2.1build1) ...
    Selecting previously unselected package libsrtp2-1:amd64.
    Preparing to unpack .../25-libsrtp2-1_2.5.0-3build1_amd64.deb ...
    Unpacking libsrtp2-1:amd64 (2.5.0-3build1) ...
    Selecting previously unselected package libvo-aacenc0:amd64.
    Preparing to unpack .../26-libvo-aacenc0_0.1.3-2build1_amd64.deb ...
    Unpacking libvo-aacenc0:amd64 (0.1.3-2build1) ...
    Selecting previously unselected package libvo-amrwbenc0:amd64.
    Preparing to unpack .../27-libvo-amrwbenc0_0.1.3-2build1_amd64.deb ...
    Unpacking libvo-amrwbenc0:amd64 (0.1.3-2build1) ...
    Selecting previously unselected package libwildmidi2:amd64.
    Preparing to unpack .../28-libwildmidi2_0.4.3-1build3_amd64.deb ...
    Unpacking libwildmidi2:amd64 (0.4.3-1build3) ...
    Selecting previously unselected package libzbar0t64:amd64.
    Preparing to unpack .../29-libzbar0t64_0.23.93-4build3_amd64.deb ...
    Unpacking libzbar0t64:amd64 (0.23.93-4build3) ...
    Selecting previously unselected package libzxing3:amd64.
    Preparing to unpack .../30-libzxing3_2.2.1-3_amd64.deb ...
    Unpacking libzxing3:amd64 (2.2.1-3) ...
    Selecting previously unselected package gstreamer1.0-plugins-bad:amd64.
    Preparing to unpack .../31-gstreamer1.0-plugins-bad_1.24.2-1ubuntu4_amd64.deb ..
    .
    Unpacking gstreamer1.0-plugins-bad:amd64 (1.24.2-1ubuntu4) ...
    Selecting previously unselected package libgstrtspserver-1.0-0.
    Preparing to unpack .../32-libgstrtspserver-1.0-0_1.24.2-1_amd64.deb ...
    Unpacking libgstrtspserver-1.0-0 (1.24.2-1) ...
    Selecting previously unselected package gnome-network-displays.
    Preparing to unpack .../33-gnome-network-displays_0.92.1-2build2_amd64.deb ...
    Unpacking gnome-network-displays (0.92.1-2build2) ...
    Selecting previously unselected package libjxr0t64:amd64.
    Preparing to unpack .../34-libjxr0t64_1.2~git20170615.f752187-5.1ubuntu2_amd64.d
    eb ...
    Unpacking libjxr0t64:amd64 (1.2~git20170615.f752187-5.1ubuntu2) ...
    Selecting previously unselected package libjxr-tools.
    Preparing to unpack .../35-libjxr-tools_1.2~git20170615.f752187-5.1ubuntu2_amd64
    .deb ...
    Unpacking libjxr-tools (1.2~git20170615.f752187-5.1ubuntu2) ...
    Selecting previously unselected package libmagickcore-6.q16-7-extra:amd64.
    Preparing to unpack .../36-libmagickcore-6.q16-7-extra_8%3a6.9.12.98+dfsg1-5.2bu
    ild2_amd64.deb ...
    Unpacking libmagickcore-6.q16-7-extra:amd64 (8:6.9.12.98+dfsg1-5.2build2) ...
    Setting up libmodplug1:amd64 (1:0.8.9.0-3build1) ...
    Setting up libvo-amrwbenc0:amd64 (0.1.3-2build1) ...
    Setting up imagemagick-6-common (8:6.9.12.98+dfsg1-5.2build2) ...
    Setting up libneon27t64:amd64 (0.33.0-1.1build3) ...
    Setting up libopenni2-0:amd64 (2.2.0.33+dfsg-18) ...
    Setting up libqrencode4:amd64 (4.1.1-1build2) ...
    Setting up libavahi-gobject0:amd64 (0.8-13ubuntu6.2) ...
    Setting up libsrtp2-1:amd64 (2.5.0-3build1) ...
    Setting up libyajl2:amd64 (2.1.0-5build1) ...
    Setting up libzbar0t64:amd64 (0.23.93-4build3) ...
    Setting up libmjpegutils-2.1-0t64:amd64 (1:2.1.0+debian-8.1build1) ...
    Setting up libvo-aacenc0:amd64 (0.1.3-2build1) ...
    Setting up libsoundtouch1:amd64 (2.3.2+ds1-1build1) ...
    Setting up libjxr0t64:amd64 (1.2~git20170615.f752187-5.1ubuntu2) ...
    Setting up libzxing3:amd64 (2.2.1-3) ...
    Setting up libopenh264-7:amd64 (2.4.1+dfsg-1) ...
    Setting up libltc11:amd64 (1.3.2-1build1) ...
    Setting up libavtp0:amd64 (0.2.0-1build1) ...
    Setting up libdirectfb-1.7-7t64:amd64 (1.7.7-11.1ubuntu2) ...
    Setting up libspandsp2t64:amd64 (0.0.6+dfsg-2.1build1) ...
    Setting up liblqr-1-0:amd64 (0.4.2-2.1build2) ...
    Setting up libwildmidi2:amd64 (0.4.3-1build3) ...
    Setting up libmpeg2encpp-2.1-0t64:amd64 (1:2.1.0+debian-8.1build1) ...
    Setting up libmfx1:amd64 (22.5.4-1) ...
    Setting up timgm6mb-soundfont (1.3-5) ...
    update-alternatives: using /usr/share/sounds/sf2/TimGM6mb.sf2 to provide /usr/sh
    are/sounds/sf2/default-GM.sf2 (default-GM.sf2) in auto mode
    update-alternatives: using /usr/share/sounds/sf2/TimGM6mb.sf2 to provide /usr/sh
    are/sounds/sf3/default-GM.sf3 (default-GM.sf3) in auto mode
    Setting up libmplex2-2.1-0t64:amd64 (1:2.1.0+debian-8.1build1) ...
    Setting up libinstpatch-1.0-2:amd64 (1.1.6-1build2) ...
    Setting up libfluidsynth3:amd64 (2.3.4-1build3) ...
    Setting up libjxr-tools (1.2~git20170615.f752187-5.1ubuntu2) ...
    Setting up libraptor2-0:amd64 (2.0.16-3ubuntu0.1) ...
    Setting up libmagickcore-6.q16-7t64:amd64 (8:6.9.12.98+dfsg1-5.2build2) ...
    Setting up libmagickwand-6.q16-7t64:amd64 (8:6.9.12.98+dfsg1-5.2build2) ...
    Setting up libmagickcore-6.q16-7-extra:amd64 (8:6.9.12.98+dfsg1-5.2build2) ...
    Setting up liblrdf0:amd64 (0.6.1-4build1) ...
    Setting up gstreamer1.0-plugins-bad:amd64 (1.24.2-1ubuntu4) ...
    Setting up libgstrtspserver-1.0-0 (1.24.2-1) ...
    Setting up gnome-network-displays (0.92.1-2build2) ...
    Processing triggers for hicolor-icon-theme (0.17-2) ...
    Processing triggers for gnome-menus (3.36.0-1.1ubuntu3) ...
    Processing triggers for libc-bin (2.39-0ubuntu8.7) ...
    Processing triggers for man-db (2.12.0-4build2) ...
    Processing triggers for desktop-file-utils (0.27-2build1) ...
    seif@seif-ASUS-TUF-Gaming-A15-FA507NVR-FA507NVR:~$

# RMSNorm

In [1]:
def rms_norm(x,eps=1e-6): # no params
    return x / ((torch.sqrt(x.square().mean(dim=-1, keepdim=True))) + eps)

In [2]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6): # dim is the embedding dimension
        super().__init__()
        self.eps = eps
        self.param = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return rms_norm(x, self.eps) * self.param

NameError: name 'nn' is not defined

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, config): # dim is the embedding dimension
        super().__init__()
        self.eps = config.eps
        self.param = nn.Parameter(torch.ones(config.n_embd))

    def forward(self, x):
        return rms_norm(x, self.eps) * self.param

# Using the RMSNorm in the attention